# BEIR SciFact Benchmark — AI King STAR v3.5

**目標**：世界第一 nDCG@10（SOTA E5-PT = 0.737）

**策略**：多路並行檢索 + RRF 融合 + Cross-encoder Rerank

**Runtime**：Colab T4 GPU（16GB VRAM）

In [ ]:
# Cell 1: Install dependencies
!pip install -q beir sentence-transformers chromadb rank-bm25 torch
!nvidia-smi

In [ ]:
# Cell 2: Imports + Config
import json
import math
import os
import time
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Optional

import torch
from beir import util
from beir.datasets.data_loader import GenericDataLoader

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: Load SciFact dataset
DATASET = "scifact"
URL = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"

data_path = util.download_and_unzip(URL, "datasets")
corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="test")
print(f"Corpus: {len(corpus)} docs | Queries: {len(queries)}")

In [ ]:
# Cell 4: Metrics
def ndcg_at_k(ranked_ids, relevant, k=10):
    dcg = sum(relevant.get(did, 0) / math.log2(i + 2) for i, did in enumerate(ranked_ids[:k]))
    ideal = sorted(relevant.values(), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def recall_at_k(ranked_ids, relevant, k=100):
    if not relevant: return 0.0
    return sum(1 for did in ranked_ids[:k] if did in relevant) / len(relevant)

def precision_at_k(ranked_ids, relevant, k=10):
    if k == 0: return 0.0
    return sum(1 for did in ranked_ids[:k] if did in relevant) / k

def mrr(ranked_ids, relevant):
    for i, did in enumerate(ranked_ids):
        if did in relevant: return 1.0 / (i + 1)
    return 0.0

def rrf(result_lists, k=60):
    scores = defaultdict(float)
    for results in result_lists:
        for rank, (did, _) in enumerate(results):
            scores[did] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("Metrics ready")

In [ ]:
# Cell 5: BM25 Index
from rank_bm25 import BM25Okapi
import re

def tokenize(text):
    return re.findall(r'\w+', text.lower())

doc_ids = list(corpus.keys())
doc_texts = [f"{corpus[did].get('title', '')} {corpus[did].get('text', '')}".strip() for did in doc_ids]
tokenized = [tokenize(t) for t in doc_texts]

t0 = time.time()
bm25 = BM25Okapi(tokenized)
print(f"BM25 built in {time.time()-t0:.1f}s ({len(doc_ids)} docs)")

def search_bm25(query, top_k=100):
    scores = bm25.get_scores(tokenize(query))
    top_idx = scores.argsort()[-top_k:][::-1]
    return [(doc_ids[i], float(scores[i])) for i in top_idx if scores[i] > 0]

In [ ]:
# Cell 6: Dense Index Builder (GPU-accelerated)
from sentence_transformers import SentenceTransformer
import chromadb
import numpy as np

class DenseSearch:
    def __init__(self, model_name, corpus, doc_ids, cache_tag=""):
        self.model_name = model_name
        self.doc_ids = doc_ids
        tag = cache_tag or model_name.replace("/", "_")
        db_path = f"dense_cache/{tag}"
        os.makedirs(db_path, exist_ok=True)
        
        print(f"Loading {model_name}...")
        self.model = SentenceTransformer(model_name, device=DEVICE)
        
        self.client = chromadb.PersistentClient(path=db_path)
        try:
            col = self.client.get_collection("dense")
            if col.count() >= len(corpus):
                self.collection = col
                print(f"  Cached ({col.count()} vectors)")
                return
        except Exception:
            pass
        
        try:
            self.client.delete_collection("dense")
        except Exception:
            pass
        self.collection = self.client.create_collection("dense", metadata={"hnsw:space": "cosine"})
        
        texts = [f"{corpus[did].get('title', '')} {corpus[did].get('text', '')}".strip() for did in doc_ids]
        
        t0 = time.time()
        batch_size = 256
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            batch_ids = doc_ids[i:i+batch_size]
            embs = self.model.encode(batch, show_progress_bar=False, batch_size=64).tolist()
            self.collection.add(ids=batch_ids, embeddings=embs, documents=batch)
            print(f"  Embedded {min(i+batch_size, len(texts))}/{len(texts)}", end="\r")
        print(f"\n  Done in {time.time()-t0:.1f}s")
    
    def search(self, query, top_k=100):
        emb = self.model.encode([query], show_progress_bar=False).tolist()
        results = self.collection.query(query_embeddings=emb, n_results=min(top_k, self.collection.count()))
        out = []
        if results and results["ids"]:
            for j, did in enumerate(results["ids"][0]):
                score = 1.0 - results["distances"][0][j] if results.get("distances") else 0.0
                out.append((did, score))
        return out

print("DenseSearch class ready")

In [ ]:
# Cell 7: Cross-encoder Reranker
from sentence_transformers import CrossEncoder

class Reranker:
    def __init__(self, model_name):
        print(f"Loading reranker: {model_name}...")
        self.model = CrossEncoder(model_name, max_length=512, device=DEVICE)
        self.name = model_name
        print("  Reranker ready")
    
    def rerank(self, query, candidates, corpus, top_k=100):
        if not candidates:
            return []
        rerank_k = min(50, len(candidates))
        top_cands = candidates[:rerank_k]
        pairs = [
            (query, corpus[did].get("title", "") + " " + corpus[did].get("text", ""))
            for did, _ in top_cands if did in corpus
        ]
        if not pairs:
            return candidates[:top_k]
        scores = self.model.predict(pairs)
        reranked = sorted(
            zip([did for did, _ in top_cands[:len(pairs)]], scores),
            key=lambda x: x[1], reverse=True
        )
        reranked_ids = {did for did, _ in reranked}
        rest = [(did, s) for did, s in candidates[rerank_k:] if did not in reranked_ids]
        return [(did, float(s)) for did, s in reranked] + rest

print("Reranker class ready")

In [ ]:
# Cell 8: Benchmark Runner
@dataclass
class BenchConfig:
    name: str
    dense_models: list = field(default_factory=list)  # list of DenseSearch
    use_bm25: bool = True
    reranker: Optional[object] = None
    top_k: int = 100

def run_benchmark(config: BenchConfig, queries, qrels, corpus):
    print(f"\n{'='*60}")
    print(f"Running: {config.name}")
    print(f"{'='*60}")
    
    all_ndcg, all_recall, all_prec, all_mrr_scores = [], [], [], []
    total_time = 0
    
    for qi, (qid, query_text) in enumerate(queries.items()):
        relevant = {did: rel for did, rel in qrels.get(qid, {}).items() if rel > 0}
        t0 = time.time()
        
        result_lists = []
        if config.use_bm25:
            result_lists.append(search_bm25(query_text, config.top_k))
        for ds in config.dense_models:
            result_lists.append(ds.search(query_text, config.top_k))
        
        if len(result_lists) > 1:
            fused = rrf(result_lists)
        elif result_lists:
            fused = result_lists[0]
        else:
            fused = []
        
        if config.reranker and fused:
            fused = config.reranker.rerank(query_text, fused, corpus, config.top_k)
        
        query_time = time.time() - t0
        total_time += query_time
        
        ranked = [did for did, _ in fused[:config.top_k]]
        all_ndcg.append(ndcg_at_k(ranked, relevant))
        all_recall.append(recall_at_k(ranked, relevant))
        all_prec.append(precision_at_k(ranked, relevant))
        all_mrr_scores.append(mrr(ranked, relevant))
        
        if (qi + 1) % 100 == 0 or qi == 0:
            print(f"  [{qi+1}/{len(queries)}] nDCG@10={all_ndcg[-1]:.3f} ({query_time*1000:.0f}ms)")
    
    results = {
        "name": config.name,
        "nDCG@10": round(sum(all_ndcg) / len(all_ndcg), 4),
        "Recall@100": round(sum(all_recall) / len(all_recall), 4),
        "P@10": round(sum(all_prec) / len(all_prec), 4),
        "MRR": round(sum(all_mrr_scores) / len(all_mrr_scores), 4),
        "Avg_ms": round(total_time / len(queries) * 1000, 1),
    }
    
    print(f"\n  nDCG@10:    {results['nDCG@10']}")
    print(f"  Recall@100: {results['Recall@100']}")
    print(f"  P@10:       {results['P@10']}")
    print(f"  MRR:        {results['MRR']}")
    print(f"  Latency:    {results['Avg_ms']} ms/query")
    
    return results

print("Runner ready")

In [ ]:
# Cell 9: Build all Dense indices (GPU — should be fast)
# Models to test (ranked by expected SciFact performance)

DENSE_MODELS = {
    "e5-large": "intfloat/e5-large-v2",
    "bge-m3": "BAAI/bge-m3",
    "bge-large": "BAAI/bge-large-en-v1.5",
}

dense_indices = {}
for tag, model_name in DENSE_MODELS.items():
    print(f"\n--- Building {tag} ---")
    dense_indices[tag] = DenseSearch(model_name, corpus, doc_ids, cache_tag=tag)

print(f"\n✅ {len(dense_indices)} dense indices ready")

In [ ]:
# Cell 10: Load Rerankers
RERANKER_MODELS = {
    "ms-marco-L6": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "bge-reranker": "BAAI/bge-reranker-v2-m3",
}

rerankers = {}
for tag, model_name in RERANKER_MODELS.items():
    rerankers[tag] = Reranker(model_name)

print(f"\n✅ {len(rerankers)} rerankers ready")

In [ ]:
# Cell 11: Run ALL configurations — Full Ablation Matrix
ALL_RESULTS = []

# === Baseline ===
ALL_RESULTS.append(run_benchmark(
    BenchConfig(name="BM25 only", use_bm25=True), queries, qrels, corpus))

# === Single Dense (each model solo) ===
for tag, ds in dense_indices.items():
    ALL_RESULTS.append(run_benchmark(
        BenchConfig(name=f"Dense:{tag}", dense_models=[ds], use_bm25=False),
        queries, qrels, corpus))

# === Hybrid: BM25 + each Dense ===
for tag, ds in dense_indices.items():
    ALL_RESULTS.append(run_benchmark(
        BenchConfig(name=f"Hybrid:BM25+{tag}", dense_models=[ds], use_bm25=True),
        queries, qrels, corpus))

# === Triple: BM25 + 2 Dense (multi-path) ===
ALL_RESULTS.append(run_benchmark(
    BenchConfig(name="Triple:BM25+e5+bge-m3",
                dense_models=[dense_indices["e5-large"], dense_indices["bge-m3"]],
                use_bm25=True),
    queries, qrels, corpus))

ALL_RESULTS.append(run_benchmark(
    BenchConfig(name="Triple:BM25+e5+bge-large",
                dense_models=[dense_indices["e5-large"], dense_indices["bge-large"]],
                use_bm25=True),
    queries, qrels, corpus))

# === Quad: BM25 + ALL 3 Dense ===
ALL_RESULTS.append(run_benchmark(
    BenchConfig(name="Quad:BM25+e5+bge-m3+bge-large",
                dense_models=list(dense_indices.values()),
                use_bm25=True),
    queries, qrels, corpus))

print(f"\n✅ {len(ALL_RESULTS)} configurations tested")

In [ ]:
# Cell 12: Add Reranker to best configs
RERANKED_RESULTS = []

# Best hybrid + each reranker
best_dense = dense_indices["e5-large"]

for rtag, reranker in rerankers.items():
    # Hybrid + reranker
    RERANKED_RESULTS.append(run_benchmark(
        BenchConfig(name=f"Hybrid:BM25+e5+{rtag}",
                    dense_models=[best_dense], use_bm25=True, reranker=reranker),
        queries, qrels, corpus))
    
    # Triple + reranker
    RERANKED_RESULTS.append(run_benchmark(
        BenchConfig(name=f"Triple:BM25+e5+bge-m3+{rtag}",
                    dense_models=[dense_indices["e5-large"], dense_indices["bge-m3"]],
                    use_bm25=True, reranker=reranker),
        queries, qrels, corpus))

ALL_RESULTS.extend(RERANKED_RESULTS)
print(f"\n✅ Total: {len(ALL_RESULTS)} configurations")

In [ ]:
# Cell 13: Final Ablation Table
print("\n" + "="*90)
print("BEIR SciFact — AI King STAR v3.5 Full Ablation")
print("="*90)
print(f"{'Config':<45} {'nDCG@10':>8} {'R@100':>8} {'P@10':>7} {'MRR':>7} {'ms':>7}")
print("-"*90)

# Sort by nDCG@10 descending
for r in sorted(ALL_RESULTS, key=lambda x: x["nDCG@10"], reverse=True):
    marker = "★" if r["nDCG@10"] >= 0.737 else " "
    print(f"{marker} {r['name']:<43} {r['nDCG@10']:>8.4f} {r['Recall@100']:>8.4f} {r['P@10']:>7.4f} {r['MRR']:>7.4f} {r['Avg_ms']:>7.1f}")

print("-"*90)
print(f"SOTA Reference: E5-PT_base = 0.737 | BM25 baseline = 0.665")
print(f"Total configs tested: {len(ALL_RESULTS)}")

In [ ]:
# Cell 14: Save results to JSON
output = {
    "team": "AI King",
    "system": "CCC STAR v3.5",
    "dataset": "scifact",
    "timestamp": time.strftime("%Y-%m-%d %H:%M"),
    "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU",
    "sota_reference": {"E5-PT_base": 0.737, "BM25": 0.665},
    "results": sorted(ALL_RESULTS, key=lambda x: x["nDCG@10"], reverse=True),
}

with open("beir_scifact_full_ablation.json", "w") as f:
    json.dump(output, f, indent=2)

# Markdown summary
lines = [
    f"# BEIR SciFact Full Ablation — AI King STAR v3.5 | {time.strftime('%Y-%m-%d')}",
    "",
    f"Device: {output['gpu']} | Configs: {len(ALL_RESULTS)}",
    "",
    "| # | Config | nDCG@10 | Recall@100 | P@10 | MRR | ms/q |",
    "|---|--------|---------|------------|------|-----|------|",
]
for i, r in enumerate(output["results"], 1):
    marker = "**" if r["nDCG@10"] >= 0.737 else ""
    lines.append(
        f"| {i} | {marker}{r['name']}{marker} | {r['nDCG@10']:.4f} | {r['Recall@100']:.4f} | {r['P@10']:.4f} | {r['MRR']:.4f} | {r['Avg_ms']:.1f} |"
    )
lines.extend(["", f"SOTA: E5-PT_base = 0.737 | BM25 = 0.665"])

with open("beir_scifact_full_ablation.md", "w") as f:
    f.write("\n".join(lines))

print("✅ Saved: beir_scifact_full_ablation.json + .md")
print("\nDownload these files to E:/Cursor/projects/tempero/benchmarks/results/")